### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
working_folder='./'

In [ ]:
# [PATCHED] !pip install datasets
# [PATCHED] !pip install evaluate

In [ ]:
from datasets import *

In [ ]:
preprocessed_train_ds = load_from_disk(working_folder  + 'preprocessed_train_dataset')
preprocessed_val_ds = load_from_disk(working_folder  + 'preprocessed_val_dataset')
preprocessed_test_ds = load_from_disk(working_folder  + 'preprocessed_test_dataset')

In [ ]:
preprocessed_train_ds

In [ ]:
preprocessed_val_ds

In [ ]:
preprocessed_test_ds

In [ ]:
model_id='google/vit-base-patch16-224-in21k'

In [ ]:
import torch
import torch.nn as nn
from transformers import ViTModel
from transformers.modeling_outputs import SequenceClassifierOutput

In [ ]:
class ViTForImageClassification(nn.Module):

    def __init__(self, num_labels=7):

        super(ViTForImageClassification, self).__init__()

        self.vit = ViTModel.from_pretrained(model_id)

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(self.vit.config.hidden_size, num_labels)

        self.num_labels = num_labels

    def forward(self, pixel_values, labels=None):

        outputs = self.vit(pixel_values=pixel_values)

        output = self.dropout(outputs.last_hidden_state[:, 0])

        logits = self.classifier(output)
        if labels is not None:

          if isinstance(labels, list):
            labels = torch.tensor(labels, dtype=torch.long, device=logits.device)

          loss_fct = nn.CrossEntropyLoss()
          pred_logits = logits.view(-1, self.num_labels)
          actual_labels = labels.view(-1)
          loss = loss_fct(pred_logits, actual_labels)

          return SequenceClassifierOutput(
              loss=loss,
              logits=logits,
              hidden_states=outputs.hidden_states,
              attentions=outputs.attentions,
          )
        else:
          return logits

In [ ]:
from transformers import TrainingArguments, Trainer

In [ ]:
metric_name = "accuracy"

args = TrainingArguments(
    output_dir= "output-fer",
    eval_strategy = "epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    logging_dir='logs',
    report_to="none"
)

In [ ]:
from evaluate import load
import numpy as np
metric = load("accuracy", trust_remote_code=True)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
model = ViTForImageClassification()

In [ ]:
trainer = Trainer(
    model = model,
    args = args,
    train_dataset = preprocessed_train_ds,
    eval_dataset = preprocessed_val_ds,
    compute_metrics = compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
outputs = trainer.predict(preprocessed_test_ds)
accuracy_result = compute_metrics((outputs.predictions, outputs.label_ids))

print("Test Accuracy:", round(100*accuracy_result["accuracy"],2))

In [ ]:
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
y_true= outputs.label_ids
y_pred=outputs.predictions.argmax(1)
cm = confusion_matrix(y_true, y_pred)

In [ ]:
classes_names = ['Anger', 'Disgust', 'Fear', 'Happiness', 'Sadness', 'Surprise', 'Neutral']

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
ax = sns.heatmap(cm, annot=True, fmt="d", linewidths=.5, xticklabels=classes_names, yticklabels=classes_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
torch.save(model, working_folder + 'ViT_fine_Tuned_FED')